## **PORTAFOLIO ACADÉMICO DE BI:**
# **PROYECTO INMOBILIARIO — CIUDAD DE LA COSTA**
# **Metadatos y Conexión — Primer análisis**
---
## Módulo 1:
* Introducción: Metadatos y Gobernanza
* Conexión y autenticación: BigQuery
* FASE I: Plan de Transformación y Corrección de Datos (ETL)
* FASE II: Análisis exploratorio de Datos (EDA)
* FASE III: Estadísticas del Dataset

---
# **Introducción**
## **Metadatos y Gobernanza:**

Antes de iniciar con el análisis mediante SQL  documentamos el origen, la estructura y los límites del dataset dándole lugar a la transparencia metodológica.


### Ficha Técnica del proyecto:
* **Nombre del dataset:** mercado_inmobiliario - ciudad_de_la_costa (proyectosuy)
* **Autoría:** Malena Rodríguez (Analista de Datos & BI)
* **Creación del dataset:** 2026/7/7 - Uruguay
* **Frecuencia de actualización:** 1 mes.
* **Origen de los datos:** Información real extraída manualmente de **venta y alquiler** en la plataforma Mercado Libre (Uruguay).

   Se omitieron completamente datos de identificación (nombres, direcciones, contacto).

   Recolección y limpieza general de datos realizada en **Google Sheets**, conexión a **BigQuery** vía URL.
 * **Destino del Análisis:** notebook académico explicativo / caso de estudio.

   Evaluación estratégica de stock, precios y detalles de propiedades en Ciudad de la Costa para optimización de cartera corporativa.

---

## Diccionario de Datos y estado de completitud:
Para la toma de decisiones estratégicas contamos con variables de alta confianza y otras que presentan omisiones:
| Nombre de columna | Tipo de dato | Descripción comercial | Estado de completitud |
| :--- | :--- | :--- | :--- |
| `id` | INTEGER | Identificador único de la publicación. | 100% Completo |
| `publicacion` | DATE | Fecha en la que se publicó el aviso. | 100% Completo |
| `finalizacion` | DATE | Fecha de baja de la plataforma (indica retiro del anuncio, NO venta). | **Incompleto** (Sesgo) |
| `operacion` | ENUM | Tipo de transacción (Venta / Alquiler). | 100% Completo |
| `tipo_inmueble` | ENUM | Categoría (Casa, Dúplex, Terreno, Complejo, etc.). | 100% Completo |
| `moneda`| ENUM | Moneda de origen (USD / UYU). | 100% Completo |  
|`precio` | FLOAT64 | Precio de lista. | 100% Completo |
| `ubicacion` | ENUM | Barrio de Ciudad de la Costa (Solymar, Lagomar, El Pinar, etc). | 100% Completo |
| `zona`| ENUM | Al norte o sur de Avenida Principal (Giannattasio) | 100% Completo |
| `pisos` / `habitaciones` / `banos` | INTEGER | Características internas declaradas en la propiedad. | 100% Completo |
| `cochera` / `parrillero` / `jardin` / `piscina` | INTEGER y BOOLEAN | Características externas declaradas en la propiedad. | 100% Completo |
| `mts2_terreno` / `mts2_edificado` | FLOAT64 | Superficie total del predio y área techada construida. | **Incompleto (Límite)** |
| `antiguedad` | INTEGER | Antigüedad de la construcción. | **Incompleto (Límite)** |
| `gastos_comunes_UYU` | FLOAT64 | Costo de mantenimiento mensual en pesos uruguayos. | **Incompleto** |
| `detalles` | STRING | Características relevantes que aumentan o bajan el valor de la propiedad. | 100% Completo |

---

### Límites del Dataset y gestión de sesgos (Gobernanza):

Para que el análisis sea matemáticamente válido y comercialmente honesto, tener en cuenta:

1. La columna `finalizacion` registra cuándo se dio de baja un aviso, **no garantiza jurídicamente una venta.** Puede deberse a la expiración del aviso, retiro por decisión del dueño o cambio de inmobiliaria. En el análisis de liquidez académica, se define esto como "Tiempo de Permanencia en Cartelera".
   
2. Las valuaciones de terrenos y grandes predios sin edificar (especialmente en zonas de alto valor comercial o cercanas al Aeropuerto de Carrasco) distorsionan gravemente los promedios de vivienda. Por lo tanto, en algunas rutas de análisis de hogares **excluimos la categoría 'terreno'** en las consultas SQL.

3. La falta de datos en metros cuadrados o gastos comunes se gestiona de forma nativa en BigQuery. SQL ignora las celdas nulas en funciones de agregación (como promedios) lo que nos permite calcular tendencias sobre muestras representativas sin necesidad de eliminar registros valiosos del total.

## **Conexión y Autenticación con el Data Warehouse (BigQuery):**

Antes de realizar cualquier consulta analítica, debemos establecer un puente seguro entre nuestro cuaderno de **Google Colab** y nuestro almacén de datos en **BigQuery**.

Ejecutaremos un proceso de autenticación oficial que utiliza la infraestructura de seguridad de Google garantizando:
1. No dejamos expuestas credenciales, contraseñas ni correos en el código público.
2. Nos conectamos en tiempo real al proyecto `proyectosuy` para consultar las tablas de forma nativa utilizando la biblioteca oficial de Google Cloud.
3. Utilizaremos la librería **Pandas**, el estándar de la industria en Python, para transformar las respuestas SQL en tablas visuales e interactivas fáciles de leer.

In [ ]:
# =============================================================================
# PASO 1: AUTENTICACIÓN CON GOOGLE CLOUD
# =============================================================================

# Desde la librería de Google Colab, importamos el módulo de autenticación (auth)
from google.colab import auth

# Ejecutamos la función que abre la ventana emergente segura de Google,
# sirve para que seleccionar la cuenta de Gmail y dar permisos de lectura a este Notebook.
auth.authenticate_user()

# Imprimimos un mensaje en pantalla para confirmar visualmente que el proceso terminó con éxito
print('Autenticación exitosa con Google Cloud')


# =============================================================================
# PASO 2: IMPORTACIÓN DE LIBRERÍAS DE TRABAJO
# =============================================================================

# Importamos el cliente oficial de BigQuery para poder enviarle consultas SQL desde Python
from google.cloud import bigquery

# Importamos Pandas (utilizamos AS para renombrarla `pd` y optimizar la estructura de las consultas).
# Sirve para convertir los datos que nos devuelva BigQuery en tablas interactivas (DataFrames).
import pandas as pd


# =============================================================================
# PASO 3: CONFIGURACIÓN DEL CLIENTE DE BIGQUERY
# =============================================================================

# Definimos en una variable de texto el ID exacto del proyecto en Google Cloud
project_id = 'proyectosuy'

# Creamos el objeto 'client' (cliente) de BigQuery usando el ID del proyecto.
# Este 'client' será el encargado de tomar nuestras consultas SQL, enviárselas a BigQuery,
# traer los resultados y entregárselos a Pandas.
client = bigquery.Client(project=project_id)

Autenticación exitosa con Google Cloud


## **FASE I: Plan de Transformación y Corrección de Datos (ETL)**

Para alinear el esquema físico de **BigQuery** con nuestro **Diccionario de Datos**, aplicamos una serie de transformaciones mediante SQL.

A continuación, detallamos los campos modificados y las reglas aplicadas:

* **publicacion (de STRING a DATE):** Convertido a formato fecha mediante `PARSE_DATE` para habilitar cálculos de tiempo y permanencia.
* **parrillero / jardin / piscina (de STRING a BOOLEAN):** Traducimos las respuestas de texto `"Si"`/`"No"` a valores lógicos `TRUE` / `FALSE` con condicionales `CASE WHEN`.
* **Estandarización de Categorías (emulación de ENUM):** Validamos y restringimos las columnas clave para asegurar que solo ingresen las opciones oficiales del proyecto, eliminando cualquier inconsistencia de mayúsculas, minúsculas o errores de carga manual:
  * **Operación:** venta, alquiler.
  * **Tipo de Inmueble:** casa, apartamento, complejo, duplex, terreno.
  * **Moneda:** UYU, U$S.
  * **Ubicación:** solymar, carrasco, pinar, lagomar, tahona, shangrila.
  * **Zona:** Sur, Norte
.

In [ ]:
# =============================================================================
# CREACIÓN DE LA VISTA EN BIGQUERY: v_ciudad_de_la_costa
# =============================================================================

# Se utiliza la instrucción "CREATE OR REPLACE VIEW" para estructurar la vista
# y asegurar la actualización del esquema de forma automatizada en el Data Warehouse.
query_vista_consolidada = """
CREATE OR REPLACE VIEW `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa` AS
SELECT
    id,

    -- 1. TRANSFORMACIÓN DE FECHA:
    -- Conversión del campo de publicación (STRING en origen) a formato DATE real
    -- para posibilitar cálculos cronológicos y análisis de permanencia.
    PARSE_DATE('%d/%m/%E4Y', publicacion) AS publicacion,

    finalizacion,
    operacion,
    tipo_inmueble,
    moneda,
    precio,
    ubicacion,
    zona,
    pisos,
    habitaciones,
    banos,
    cochera,

    -- 2. TRADUCCIÓN A CAMPOS BOOLEANOS:
    -- Conversión de las respuestas cualitativas "Si"/"No" a tipos lógicos nativos (TRUE/FALSE)
    -- optimizando el rendimiento de la base de datos y su lectura en herramientas de BI.
    CASE WHEN LOWER(parrillero) = 'si' THEN TRUE ELSE FALSE END AS parrillero,
    CASE WHEN LOWER(jardin) = 'si' THEN TRUE ELSE FALSE END AS jardin,
    CASE WHEN LOWER(piscina) = 'si' THEN TRUE ELSE FALSE END AS piscina,

    mts2_terreno,
    mts2_edificado,
    antiguedad,
    gastos_comunes_UYU,
    detalles
FROM
    `proyectosuy.mercado_inmobiliario.ciudad_de_la_costa`
WHERE
    -- 3. REGLAS DE GOBERNANZA (Emulación de ENUM y normalización de texto):
    -- Aplicación de la función LOWER() para estandarizar las entradas en minúsculas,
    -- garantizando la integridad de las comparaciones frente a variaciones de tipeo.
    LOWER(operacion) IN ('venta', 'alquiler')
    AND LOWER(tipo_inmueble) IN ('casa', 'apartamento', 'complejo', 'duplex', 'terreno')
    AND LOWER(moneda) IN ('uyu', 'u$s')
    AND LOWER(zona) IN ('sur', 'norte')
    AND LOWER(ubicacion) IN ('solymar', 'carrasco', 'pinar', 'lagomar', 'tahona', 'shangrila');
"""

# Ejecución de la consulta a través del cliente autenticado de BigQuery
client.query(query_vista_consolidada)

# Confirmación de salida en el entorno de ejecución
print("Vista 'v_ciudad_de_la_costa' creada y consolidada con todas las opciones de calidad")

Vista 'v_ciudad_de_la_costa' creada y consolidada con todas las opciones de calidad


Con la creación exitosa de nuestra vista `v_ciudad_de_la_costa`, damos por finalizada la fase de **Extracción, Transformación y Carga (ETL)** e infraestructura de datos.

Antes de proceder a la fase de análisis exploratorio, ejecutaremos una consulta de **validación final** para confirmar el volumen de datos consolidados.

In [ ]:
# Definimos una consulta rápida para contar el total de filas que sobrevivieron
# a las reglas de gobernanza y traer una muestra de los primeros 3 registros.
query_verificacion = """
SELECT
    (SELECT COUNT(*) FROM `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`) AS total_registros,
    id,
    publicacion,
    operacion,
    tipo_inmueble,
    precio,
    moneda,
    ubicacion,
    parrillero,
    piscina
FROM
    `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
LIMIT 3;
"""

# Ejecutamos la consulta y la cargamos directamente en un DataFrame de Pandas
df_resultado = client.query(query_verificacion).to_dataframe()

# Desplegamos el resultado en el Notebook
df_resultado

,total_registros,id,publicacion,operacion,tipo_inmueble,precio,moneda,ubicacion,parrillero,piscina
0,555,1,2025-06-01,Venta,Casa,289000.0,U$S,Solymar,True,False
1,555,2,2026-05-07,Venta,Casa,250000.0,U$S,Solymar,True,False
2,555,3,2026-05-05,Venta,Complejo,187000.0,U$S,Solymar,True,False


En la consulta de validación anterior se realizó una **proyección selectiva** seleccionando las columnas modificadas, esta simplificación visual tuvo como objetivo validar de manera clara, directa y eficiente que las reglas de transformación (ETL) y gobernanza se aplicaron de forma correcta sobre la vista de datos, evitando la sobrecarga visual de columnas no modificadas en la pantalla del Notebook.

Habiendo verificado que la estructura responde perfectamente y que contamos con **555 registros** listos y consistentes, damos por cerrada la fase de preparación de datos.

---

## **FASE II: Análisis exploratorio de Datos (EDA)**

### Contextualización y segmentación geográfica:

Para dar inicio a nuestro análisis exploratorio comenzaremos aplicando un **enfoque segmentado por ubicación**.

Cada barrio posee dinámicas de mercado, perfiles de construcción y niveles de demanda muy particulares, por lo tanto, estructuraremos este primer bloque analítico para definir las métricas clave de cada zona.
Antes de cruzar variables generales, es fundamental comprender el ecosistema local de Ciudad de la Costa de manera fraccionada, analizando no solo las variables comerciales sino también la **calidad de los datos** disponibles por ubicación.

En el mercado inmobiliario, mezclar fracciones de tierra vacía (terrenos) con propiedades edificadas (casas, apartamentos, etc) en un mismo promedio altera de forma artificial el valor real de la zona. Por lo tanto, desde el inicio realizaremos una distinción:
* **Terrenos:** Análisis exclusivo del valor de la tierra.
* **Viviendas:** Análisis de propiedades edificadas (casas, apartamentos, complejos, duplex).

### Métricas iniciales a analizar para dar contexto a los barrios:
1. **Volumen de Stock por Barrio:** Identificar dónde se concentra la mayor oferta inmobiliaria actual de nuestra cartera.
2. **Distribución de Tipos de Inmuebles:** Analizar si las zonas son predominantemente residenciales (casas/apartamentos), si contienen desarrollos privados (complejos) o si destacan por la disponibilidad de suelo (terrenos).
3. **Precios Medios de Entrada:** Establecer la barrera económica promedio de cada ubicación para comprender el posicionamiento de mercado.
4. **Índice de Equipamiento y Confort:** Medir el porcentaje de propiedades por barrio que cuentan con garage/cochera, parrillero y piscina, definiendo el perfil cualitativo de la oferta.
5. **Auditoría de Calidad (Completitud de Metros Cuadrados):** Dado que existen omisiones en el registro de superficies, calcularemos el porcentaje exacto de registros por barrio que poseen datos válidos (no nulos) de `mts2_terreno` y `mts2_edificado`. Esto nos indicará en qué zonas es viable realizar análisis de precios por metro cuadrado.



In [ ]:
# =============================================================================
# ANÁLISIS GENERAL DE MERCADO
# =============================================================================

# Definimos la consulta SQL
# para procesar todos los barrios de la base de datos.
query_barrios_consolidado = """
SELECT
    ubicacion AS Barrio,

    -- 1. Volumen de Stock General
    COUNT(*) AS Total_Propiedades,
    COUNTIF(LOWER(tipo_inmueble) != 'terreno') AS Total_Viviendas,
    COUNTIF(LOWER(tipo_inmueble) = 'terreno') AS Total_Terrenos,

    -- 2. Distribución de Tipologías Construidas
    COUNTIF(LOWER(tipo_inmueble) = 'casa') AS Casas,
    COUNTIF(LOWER(tipo_inmueble) = 'apartamento') AS Apartamentos,
    COUNTIF(LOWER(tipo_inmueble) = 'complejo') AS Complejos,
    COUNTIF(LOWER(tipo_inmueble) = 'duplex') AS Duplex,

    -- 3. Precios Medios Separados (Venta de Terrenos vs. Venta de Viviendas)
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) = 'terreno' AND LOWER(moneda) = 'u$s' AND LOWER(operacion) = 'venta' THEN precio END), 0) AS Promedio_Venta_Terrenos_USD,
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'u$s' AND LOWER(operacion) = 'venta' THEN precio END), 0) AS Promedio_Venta_Viviendas_USD,

    -- 4. Precios Medios de Alquiler para Viviendas (Segmentado por Moneda)
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'u$s' AND LOWER(operacion) = 'alquiler' THEN precio END), 0) AS Promedio_Alquiler_Viviendas_USD,
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'uyu' AND LOWER(operacion) = 'alquiler' THEN precio END), 0) AS Promedio_Alquiler_Viviendas_UYU,

    -- 5. Índice de Equipamiento de Viviendas (% basado únicamente en propiedades construidas)
    ROUND(SAFE_DIVIDE(COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND cochera > 0), COUNTIF(LOWER(tipo_inmueble) != 'terreno')) * 100, 1) AS Porcentaje_Cochera_Viviendas,
    ROUND(SAFE_DIVIDE(COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND parrillero = TRUE), COUNTIF(LOWER(tipo_inmueble) != 'terreno')) * 100, 1) AS Porcentaje_Parrillero_Viviendas,
    ROUND(SAFE_DIVIDE(COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND piscina = TRUE), COUNTIF(LOWER(tipo_inmueble) != 'terreno')) * 100, 1) AS Porcentaje_Piscina_Viviendas,

    -- 6. Auditoría de Calidad (% de completitud de metros cuadrados para Viviendas)
    ROUND(SAFE_DIVIDE(COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND mts2_terreno IS NOT NULL), COUNTIF(LOWER(tipo_inmueble) != 'terreno')) * 100, 1) AS Completitud_M2_Terreno_Viviendas,
    ROUND(SAFE_DIVIDE(COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND mts2_edificado IS NOT NULL), COUNTIF(LOWER(tipo_inmueble) != 'terreno')) * 100, 1) AS Completitud_M2_Edificado_Viviendas
FROM
    `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
GROUP BY
    ubicacion
ORDER BY
    Total_Propiedades DESC;
"""

# Ejecutamos la consulta y la cargamos en el DataFrame de Pandas
df_barrios_consolidado = client.query(query_barrios_consolidado).to_dataframe()

# Desplegamos el resultado completo
df_barrios_consolidado

,Barrio,Total_Propiedades,Total_Viviendas,Total_Terrenos,Casas,Apartamentos,Complejos,Duplex,Promedio_Venta_Terrenos_USD,Promedio_Venta_Viviendas_USD,Promedio_Alquiler_Viviendas_USD,Promedio_Alquiler_Viviendas_UYU,Porcentaje_Cochera_Viviendas,Porcentaje_Parrillero_Viviendas,Porcentaje_Piscina_Viviendas,Completitud_M2_Terreno_Viviendas,Completitud_M2_Edificado_Viviendas
0,Solymar,314,258,56,154,13,33,58,473589.0,276716.0,NaN,34435.0,82.9,79.5,11.2,88.0,92.6
1,Shangrila,63,58,5,35,4,2,17,426000.0,302125.0,NaN,43415.0,75.9,74.1,8.6,82.8,91.4
2,Tahona,62,37,25,33,0,2,2,371676.0,650033.0,3514.0,NaN,89.2,97.3,75.7,91.9,94.6
3,Carrasco,44,24,20,9,2,3,10,2328293.0,312314.0,NaN,43667.0,79.2,79.2,20.8,83.3,91.7
4,Pinar,37,31,6,22,3,3,3,558333.0,245444.0,NaN,27531.0,80.6,45.2,12.9,74.2,90.3
5,Lagomar,35,32,3,14,1,4,13,2213333.0,294320.0,NaN,39286.0,93.8,90.6,9.4,93.8,100.0


### **Diagnóstico: Panorama general por barrio**

**Solymar (314 propiedades)**: El mercado más grande y líquido de la zona (perfil masivo) cuenta con un mix de dúplex y apartamentos. Precio promedio de vivienda USD 276,716 (el punto de entrada más accesible con mayor volumen de stock).

**La Tahona (62 propiedades):** Segmento premium confirmado. Precio promedio de vivienda USD 650,033 (más del doble del resto) 75.7% de las viviendas con piscina, y es el único barrio donde el alquiler se cotiza en USS.

**Shangrilá (63 propiedades):** Mercado estable, precio promedio USD 302,125, con buena completitud de datos (91.4% en m2 edificado).

**Carrasco (44 propiedades):** Precio de terreno promedio USD 2,328,293, pero con muestra chica (solo 20 terrenos), cifra a tomar con cautela ya que probablemente esté inflada por macro-lotes.

**Pinar (37 propiedades):** El más económico en vivienda (USD 245,444) y con la menor completitud de datos de todos (74.2% en m2 terreno). Cualquier conclusión sobre este barrio debe marcarse con ese margen de incertidumbre.

**Lagomar (35 propiedades):** El barrio con menor stock de terrenos (solo 3), y precio promedio de terreno USD 2,213,333 (mismo caso que Carrasco, cifra poco representativa por bajo N).

#### **En los seis barrios, el alquiler se cotiza en pesos uruguayos excepto en La Tahona (exclusivamente en dólares). Es la señal más clara de segmentación de mercado por poder adquisitivo.**

-----

## **FASE III: Estadísticas del Dataset**

Consolidaremos una **métrica de control de calidad** crítica para el desarrollo del proyecto.
## Alcance y Limitaciones de la Muestra

Antes de adentrarnos en el comportamiento específico de cada ubicación es indispensable tener en cuenta el origen de nuestra base de datos y definir sus límites metodológicos para no caer en sesgos de sobregeneralización:

1. **Tamaño de la Muestra:**
   Nuestra base de datos actual cuenta con un registro de **555 propiedades** distribuidas a lo largo de la Ciudad de la Costa. Si bien es un volumen sumamente robusto para un análisis exploratorio, representa un recorte específico del mercado y no la totalidad del universo inmobiliario real.

2. **Sesgo de Plataforma:**
   Los datos han sido extraídos exclusivamente de **MercadoLibre Uruguay**. Aunque esta plataforma es el portal de comercio electrónico y búsqueda de inmuebles más popular y con mayor tráfico del país, debemos considerar sus limitaciones: el mercado informal, la oferta exclusiva B2B, precios de publicación vs. precios de cierre.

Por lo tanto, este estudio debe ser interpretado como un **Análisis de la oferta Digital en el portal líder**, lo cual sigue siendo una herramienta de toma de decisiones sumamente potente, pero complementaria a la realidad del territorio.


Para poder calcular en las próximas fases el valor real del suelo y de la construcción mediante el ratio de costo por m2, necesitamos tener la certeza de que contamos con una muestra estadísticamente representativa.

A continuación, presentamos la tabla resumen que detalla exclusivamente el porcentaje de datos completos (registros no nulos) para las variables de superficie por ubicación.

In [ ]:
# =============================================================================
# COMPLETITUD DE METROS CUADRADOS POR UBICACIÓN
# =============================================================================

# Generamos una consulta enfocada exclusivamente en evaluar la calidad de los datos
# de superficie (terreno y edificado) para inmuebles construidos en cada barrio.
query_auditoria_calidad = """
SELECT
    ubicacion AS Barrio,
    COUNTIF(LOWER(tipo_inmueble) != 'terreno') AS Total_Viviendas_Analizadas,

    -- Porcentaje de registros con datos de superficie declarados
    CONCAT(ROUND((COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND mts2_terreno IS NOT NULL) / COUNTIF(LOWER(tipo_inmueble) != 'terreno')) * 100, 1), '%') AS Completitud_M2_Terreno,
    CONCAT(ROUND((COUNTIF(LOWER(tipo_inmueble) != 'terreno' AND mts2_edificado IS NOT NULL) / COUNTIF(LOWER(tipo_inmueble) != 'terreno')) * 100, 1), '%') AS Completitud_M2_Edificado
FROM
    `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
GROUP BY
    ubicacion
ORDER BY
    Total_Viviendas_Analizadas DESC;
"""

# Ejecutamos la consulta en BigQuery y la volcamos en un DataFrame de Pandas
df_auditoria = client.query(query_auditoria_calidad).to_dataframe()

# Visualizamos la tabla limpia de auditoría
df_auditoria

,Barrio,Total_Viviendas_Analizadas,Completitud_M2_Terreno,Completitud_M2_Edificado
0,Solymar,258,88%,92.6%
1,Shangrila,58,82.8%,91.4%
2,Tahona,37,91.9%,94.6%
3,Lagomar,32,93.8%,100%
4,Pinar,31,74.2%,90.3%
5,Carrasco,24,83.3%,91.7%



## Confiabilidad y Significancia Estadística

Para validar si los promedios y métricas que calcularemos en las siguientes fases son confiables para la toma de decisiones, debemos analizar la tabla de auditoría bajo tres pilares de la teoría estadística:

#### 1. Robustez de la Muestra (Margen de Error y $N$)
En estadística, el tamaño de la muestra ($N$) define el **margen de error**.
* **Solymar ($N=258$):** Cuenta con una muestra masiva. Estadísticamente, un volumen tan alto con más del 88% de completitud nos permite inferir el comportamiento del mercado con un nivel de confianza superior al 95% y un margen de error mínimo (menor al 5%). Es un grupo de datos de extrema robustez.
* **Segmento Intermedio ($N$ entre 24 y 58):** Para barrios como Shangrilá, Carrasco, Lagomar o El Pinar, aunque las muestras son más pequeñas, las tasas de completitud de Datos son sobresalientes (todas por encima del 90% en metros edificados).
#### 2. Sesgo de Información Sistemático
Un hallazgo estadístico es que **la completitud de `mts2_edificado` es sistemáticamente mayor que la de `mts2_terreno` en todos los barrios** (siempre superando el 90%, llegando al 100% en Lagomar).
Esto responde a un **sesgo de comportamiento del mercado**: los propietarios y agentes inmobiliarios consideran que la superficie construida es la variable más determinante para fijar el precio de venta, por lo que casi nunca omiten este dato al registrar una propiedad en cartelera.

#### 3. Diagnóstico de Confiabilidad por Zona
* **Solymar:** Con un **92.6% de completitud** en metros edificados sobre una muestra masiva de 258 viviendas, este barrio ofrece el nivel de confiabilidad estadística más sólido de todo el proyecto.
* **Lagomar:** Con un **100% de datos completos** en metros edificados sobre 32 viviendas, este barrio ofrece una precisión perfecta en disponibilidad de Datos.
* **La Tahona:** Registra un **94.6% de completitud**, lo que sumado a la homogeneidad de sus propiedades (viviendas de alta gama dentro de un barrio cerrado), nos asegura que los promedios no estarán distorsionados por valores atípicos (*outliers*).
* **Carrasco:** Registra un **91.7% de completitud** en metros edificados, pero la muestra es de 24 viviendas.
* **El Pinar:** Es la única zona donde la completitud del suelo desciende al **74.2%**. Si bien sigue siendo un porcentaje estadísticamente aceptable para trabajar, cualquier conclusión sobre el valor del metro cuadrado de terreno en El Pinar debe manejarse con mayor prudencia que en el resto de los barrios.

----------------------------------------------------------------------------


## **Media y Mediana:**

Al analizar los precios de venta de viviendas y terrenos, es fundamental introducir una estadística crucial para el análisis financiero: **la diferencia entre la Media (Promedio) y la Mediana.**

#### 1. La Media Aritmética (Promedio)
 Es la suma de todos los precios dividida por la cantidad total de propiedades, extremadamente sensible a los valores atípicos (*outliers*). Si en un barrio residencial de clase media con casas de USD 200,000 se publica una sola mansión de USD 2,500,000 el promedio general se elevará de forma artificial, dando la falsa impresión de que todo el barrio es mucho más costoso de lo que realmente es.

#### 2. La Mediana (Valor Central)
* Es el precio que se ubica exactamente en el medio de la muestra cuando ordenamos todas las propiedades de menor a mayor. El 50% de las propiedades cuesta menos que ese valor, y el otro 50% cuesta más.
Es inmune a los valores extremos, no importa si la casa más cara del barrio cuesta USD 500,000 o USD 5,000,000; la mediana seguirá mostrando el precio de la "vivienda típica" de la zona.

#### Conclusión Metodológica para el Proyecto:

* **La Media (Promedio):** Para entender el volumen total de capital que se mueve en la zona y calcular ratios globales (como el promedio de costo de construcción).
* **La Mediana:** Para calcular el "precio de entrada real" para un comprador común, asegurándonos de que nuestras proyecciones no estén sesgadas por propiedades excepcionalmente caras o baratas.

In [ ]:
# =============================================================================
#  MEDIA Y MEDIANA
# =============================================================================

query_media_mediana = """
WITH datos_preparados AS (
    SELECT
        ubicacion AS Barrio,
        tipo_inmueble,
        moneda,
        operacion,
        precio,

        -- Calculamos las medianas utilizando funciones analíticas de ventana completas
        PERCENTILE_CONT(
            CASE WHEN LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'u$s' AND LOWER(operacion) = 'venta' THEN precio END,
            0.5
        ) OVER(PARTITION BY ubicacion) AS Mediana_Venta_Viviendas_USD,

        PERCENTILE_CONT(
            CASE WHEN LOWER(tipo_inmueble) = 'terreno' AND LOWER(moneda) = 'u$s' AND LOWER(operacion) = 'venta' THEN precio END,
            0.5
        ) OVER(PARTITION BY ubicacion) AS Mediana_Venta_Terrenos_USD
    FROM
        `proyectosuy.mercado_inmobiliario.v_ciudad_de_la_costa`
)
SELECT
    Barrio,
    COUNT(*) AS Total_Propiedades,

    -- 1. Promedios (Media Aritmética) para Viviendas
    -- Mismo filtro de moneda y operación que usa la mediana, para no mezclar
    -- ventas en USD con alquileres en UYU dentro del mismo promedio.
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) != 'terreno' AND LOWER(moneda) = 'u$s' AND LOWER(operacion) = 'venta' THEN precio END), 0) AS Media_Venta_Viviendas_USD,

    -- Tomamos el valor de la mediana calculado en la ventana anterior (usamos MAX para resolver la agregación)
    ROUND(MAX(Mediana_Venta_Viviendas_USD), 0) AS Mediana_Venta_Viviendas_USD,

    -- 2. Promedios (Media Aritmética) para Terrenos
    ROUND(AVG(CASE WHEN LOWER(tipo_inmueble) = 'terreno' AND LOWER(moneda) = 'u$s' AND LOWER(operacion) = 'venta' THEN precio END), 0) AS Media_Venta_Terrenos_USD,
    ROUND(MAX(Mediana_Venta_Terrenos_USD), 0) AS Mediana_Venta_Terrenos_USD
FROM
    datos_preparados
GROUP BY
    Barrio
ORDER BY
    Total_Propiedades DESC;
"""

# Ejecutamos la consulta corregida en BigQuery
df_media_mediana = client.query(query_media_mediana).to_dataframe()

# Visualizamos la tabla consolidada final en el Notebook
df_media_mediana

,Barrio,Total_Propiedades,Media_Venta_Viviendas_USD,Mediana_Venta_Viviendas_USD,Media_Venta_Terrenos_USD,Mediana_Venta_Terrenos_USD
0,Solymar,314,276716.0,258000.0,473589.0,215000.0
1,Shangrila,63,302125.0,305000.0,426000.0,265000.0
2,Tahona,62,650033.0,590000.0,371676.0,215000.0
3,Carrasco,44,312314.0,295000.0,2328293.0,950000.0
4,Pinar,37,245444.0,252500.0,558333.0,331000.0
5,Lagomar,35,294320.0,270000.0,2213333.0,1300000.0


## **Diagnóstico Estadístico: Media y Mediana**
Al contrastar la Media (promedio general) con la Mediana (el valor central exacto) de nuestro stock, podemos extraer conclusiones comerciales de gran valor estratégico:

**1. La Tahona: el segmento de lujo absoluto**

La media se sitúa en USD 650,033, mientras que la mediana es de USD 590,000.
La media está por encima de la mediana, lo cual es el patrón esperable en un segmento premium: un grupo de propiedades de muy alto valor empuja el promedio hacia arriba, mientras que la "vivienda típica" del barrio (el precio de entrada real para un comprador) se ubica más cerca de los USD 590,000.

**2. Shangrilá: estabilidad y homogeneidad**

Media de USD 302,125 y mediana de USD 305,000, valores prácticamente idénticos.
Esto revela un mercado con muy poca dispersión de precios: la oferta se concentra de forma consistente en un mismo rango de valor, sin *outliers* que distorsionen el promedio. Es una señal de homogeneidad en el perfil de propiedades del barrio.

**3. El Pinar: la excepción a la regla**

Es el único barrio donde la mediana (USD 252,500) supera levemente a la media (USD 245,444).
Aunque la diferencia es mínima, sugiere una leve concentración de propiedades más económicas dentro de la oferta, que empujan el promedio ligeramente hacia abajo respecto al valor central. Vale la pena monitorear este comportamiento a medida que crezca la muestra.

**4. El sesgo clásico en terrenos: Carrasco y Lagomar**

Carrasco (Terrenos): media de USD 2,328,293 frente a una mediana de USD 950,000.

Lagomar (Terrenos): media de USD 2,213,333 frente a una mediana de USD 1,300,000.

En ambos casos la media más que duplica a la mediana, lo que demuestra de forma clara la presencia de valores atípicos extremos (probablemente macro-lotes destinados a desarrollos inmobiliarios de gran escala) que distorsionan por completo el promedio.

El precio de entrada real para un lote típico en estos barrios está mucho más cerca de la mediana que de la media: en Carrasco, más cercano a los USD 950,000 que a los más de 2 millones que sugiere el promedio; en Lagomar, más cercano a los USD 1,300,000 que a los USD 2,2 millones.

**5. Solymar: el mismo patrón, a menor escala**

Solymar (Terrenos): media de USD 473,589 frente a una mediana de USD 215,000.
Aunque el barrio tiene el volumen de stock más grande de todo el dataset (314 propiedades), también muestra una brecha considerable entre media y mediana en terrenos, reafirmando que este fenómeno de *outliers* no es exclusivo de las zonas más caras, sino un patrón recurrente en la categoría "terreno" en general.

---

### **Cierre del Módulo I:**
En este módulo documentamos la ficha técnica y Gobernanza del Dataset, establecimos la conexión a BigQuery y transformamos los tipos de datos originales mediante SQL para habilitar el análisis.

Con esa base, realizamos un primer análisis exploratorio por ubicación y evaluamos la confiabilidad estadística de la muestra, incorporando Media y Mediana para detectar distorsiones por *outliers*.

## **Próximo Módulo:**
 Rentabilidad e inversión.

 ---
Análisis realizado en Julio/2026 con fines académicos.
 ---